# F2.3 · Limpieza y manejo de valores NA

Se corrige lo que encontró la exploración, con una decisión justificada por variable. Ninguna fila se elimina.

In [1]:
# Configuración común: rutas del proyecto y módulos del pipeline.
import pandas as pd

from pipeline import chilecompra as cc
from pipeline import config, ordenes

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.1f}".format)
print("Raíz del proyecto:", config.PROJECT_ROOT.name)

Raíz del proyecto: 002_sumativa-1-avance-proyecto-f1-y-f2-40


## 1. Diagnóstico de valores NA

`normalize_missing` recorta los espacios y convierte los dos marcadores de ausencia en NA reales.

In [2]:
raw = ordenes.load_orders(config.RAW_DATA_DIR)
clean = ordenes.normalize_missing(raw)

print("NA reales antes:", int(raw.isna().sum().sum()), "| después:", f"{int(clean.isna().sum().sum()):,}")
key_columns = ["EsTratoDirecto", "EsCompraAgil", "CodigoLicitacion", "FechaAceptacion", "MontoTotalOC_PesosChilenos", "CodigoProveedor", "CodigoUnidadCompra", "RegionProveedor"]
clean[key_columns].isna().groupby(clean["periodo"]).mean().T.mul(100).round(1)

NA reales antes: 0 | después: 184,260


periodo,2024-09,2025-09
EsTratoDirecto,0.0,38.0
EsCompraAgil,0.0,0.0
CodigoLicitacion,66.9,67.4
FechaAceptacion,2.2,1.7
MontoTotalOC_PesosChilenos,0.0,0.0
CodigoProveedor,0.0,0.0
CodigoUnidadCompra,0.0,0.0
RegionProveedor,16.5,15.7


## 2. Estrategia por variable

| Variable | NA | Decisión | Justificación |
|---|---|---|---|
| `EsTratoDirecto` | 38 % en 2025 | **Conservar** el NA y resolverlo con `EsCompraAgil` | No es un dato perdido: son las compras ágiles, que desde 2025 ya no se clasifican como trato directo (se comprueba abajo) |
| `CodigoLicitacion` | 67 % | **Conservar** | Vacío significa que la orden no viene de una licitación; es información |
| `FechaAceptacion` | 2 % | **Conservar** | La orden aún no ha sido aceptada por el proveedor; no se usa en los indicadores |
| `RegionProveedor` y otros datos del proveedor | hasta 15 % | **Conservar**, sin imputar | Inventar una región distorsionaría un análisis posterior por territorio |
| Monto, proveedor y unidad de compra | 0 % | Nada que corregir | Son las variables de los indicadores y están completas |

**Enfoque descartado:** eliminar las filas con NA. Habríamos perdido todas las compras ágiles de 2025, justo una de las modalidades que estudiamos.

In [3]:
# Comprobación: los NA de EsTratoDirecto en 2025 corresponden a compras ágiles.
na_2025 = clean[(clean["periodo"] == "2025-09") & clean["EsTratoDirecto"].isna()]
print(f"Líneas sin dato de trato directo en 2025: {len(na_2025):,}")
print(na_2025["Tipo"].value_counts().to_dict(), "(AG = compra ágil)")
print(f"Son compra ágil: {(na_2025['EsCompraAgil'] == 'Si').mean():.1%}")

Líneas sin dato de trato directo en 2025: 5,270
{'AG': 5270} (AG = compra ágil)
Son compra ágil: 100.0%


## 3. Duplicados e inconsistencias

In [4]:
print("Filas completamente repetidas:", clean.duplicated().sum())
print("Líneas con el mismo IDItem:", clean["IDItem"].duplicated().sum())

# El monto total debe ser el mismo en todas las líneas de una orden.
distinct_amounts = clean.groupby("Codigo")["MontoTotalOC_PesosChilenos"].nunique()
print("Órdenes con montos totales contradictorios:", (distinct_amounts > 1).sum())
print("Órdenes que aparecen en los dos períodos:", (clean.groupby("Codigo")["periodo"].nunique() > 1).sum())

Filas completamente repetidas: 0
Líneas con el mismo IDItem: 0
Órdenes con montos totales contradictorios: 0
Órdenes que aparecen en los dos períodos: 0


In [5]:
# Los espacios sobrantes creaban categorías distintas para un mismo valor.
print("Antes: ", raw["RegionUnidadCompra"].unique().tolist())
print("Después:", clean["RegionUnidadCompra"].unique().tolist())

Antes:  ['Región de Valparaíso ', 'Valparaíso']
Después: ['Región de Valparaíso', 'Valparaíso']


## 4. Resultado intermedio verificado

In [6]:
assert len(clean) == len(raw), "La limpieza no debe eliminar filas"
assert clean["Codigo"].nunique() == raw["Codigo"].nunique()
assert not clean[["MontoTotalOC_PesosChilenos", "CodigoProveedor", "CodigoUnidadCompra"]].isna().any().any()
assert not clean.isin(ordenes.MISSING_MARKERS).any().any(), "Quedan marcadores de ausencia sin convertir"

pd.DataFrame({"antes": [len(raw), raw["Codigo"].nunique(), int(raw.isna().sum().sum())], "después": [len(clean), clean["Codigo"].nunique(), int(clean.isna().sum().sum())]}, index=["líneas", "órdenes", "valores NA reconocidos"])

,antes,después
líneas,25880,25880
órdenes,10285,10285
valores NA reconocidos,0,184260
